# Blind annotation assignment sheets

Build reproducible A4 annotation sheets from the S3 scatterplots and S4 metrics. The annotator-facing pages contain only scatterplots and anonymous plot codes such as `A1`-`A100`. Research metadata is exported separately and must not be shared with annotators.

Default design: 14 strong signal families x 16 requested SNR levels x 2 repeats, plus the same 32 reproducibly selected Null controls used by D_S6 = 480 unique cases. Cohort 1 (A-E), cohort 2 (F-J), and cohort 3 (K-O) each independently cover all 480 cases once and independently select 20 stratified random cases for duplicate annotation. Every annotator receives 100 plots in one two-page A4 landscape PDF (50 plots per page). Existing A-J assignments remain fixed because their seeds are unchanged.

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, milp

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

def locate_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in candidates:
        if (root / 'D' / 'Global' / 'D_S3_snr_sweep_generation.ipynb').exists():
            return root
    raise FileNotFoundError('Could not locate D/Global/D_S3_snr_sweep_generation.ipynb')

REPO_ROOT = locate_repo_root()
GLOBAL_DIR = REPO_ROOT / 'D' / 'Global'
S3_OUTPUT_DIR = GLOBAL_DIR / 'output' / 'S3_snr_mic_sweep'
OUTPUT_DIR = GLOBAL_DIR / 'output' / 'S7_annotation_sheets'
PDF_DIR = OUTPUT_DIR / 'pdf'
PREVIEW_DIR = OUTPUT_DIR / 'preview_png'
TEMPLATE_DIR = OUTPUT_DIR / 'annotator_label_templates'
RESEARCHER_DIR = OUTPUT_DIR / 'researcher_only'
for directory in [PDF_DIR, PREVIEW_DIR, TEMPLATE_DIR, RESEARCHER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CASES_PATH = S3_OUTPUT_DIR / 'cases.csv'
POINTS_PATH = S3_OUTPUT_DIR / 'scatter_points.npz'
METRICS_PATH = S3_OUTPUT_DIR / 'metrics_full.parquet'
for required_path in [CASES_PATH, POINTS_PATH, METRICS_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing {required_path}. Run D_S3 and D_S4 first.')

print(f'Cases:   {CASES_PATH}')
print(f'Points:  {POINTS_PATH}')
print(f'Metrics: {METRICS_PATH}')
print(f'Output:  {OUTPUT_DIR}')

Cases:   /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S3_snr_mic_sweep/cases.csv
Points:  /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S3_snr_mic_sweep/scatter_points.npz
Metrics: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S3_snr_mic_sweep/metrics_full.parquet
Output:  /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets


## Controls

All three cohorts are generated together. Cohort 1 contains A-E, cohort 2 contains F-J, and cohort 3 contains K-O. Within each cohort, all 480 cases are covered once and 20 additional cases are independently selected by stratified random sampling. Each annotator receives 100 cases, split over two printable A4 landscape pages.

In [2]:
FAMILY_SPECS = [
    ('F01', 'Linear'), ('F03', 'Power convex'), ('F05', 'Power concave'),
    ('F07', 'Saturation'), ('F13', 'S-curve'), ('F15', 'Threshold'),
    ('F18', 'U-shape'), ('F21', 'Cubic'), ('F19', 'Spike'),
    ('F23', 'Two Lines'), ('F24', 'Line + Parabola'),
    ('F25', 'Multi-regime'), ('F26', 'Windowed J'), ('F22', 'Oscillation'),
]
REQUESTED_SNR = [100.0, 20.0, 15.0, 10.0, 7.0, 5.0, 4.0, 3.0,
                 2.0, 1.5, 1.0, 0.8, 0.5, 0.4, 0.3, 0.1]
SELECTED_VARIANT = 'strong'
SELECTED_REPEATS = [1, 2]
N_NULL = 32
PLOTS_PER_ANNOTATOR = 100

COHORT_SPECS = [
    {
        'cohort_id': 'cohort1',
        'annotators': ['A', 'B', 'C', 'D', 'E'],
        'overlap_seed': 2026072102,
        'allocation_seed': 2026072103,
        'display_seed': 2026072104,
    },
    {
        'cohort_id': 'cohort2',
        'annotators': ['F', 'G', 'H', 'I', 'J'],
        'overlap_seed': 2026072202,
        'allocation_seed': 2026072203,
        'display_seed': 2026072204,
    },
    {
        'cohort_id': 'cohort3',
        'annotators': ['K', 'L', 'M', 'N', 'O'],
        'overlap_seed': 2026072302,
        'allocation_seed': 2026072303,
        'display_seed': 2026072304,
    },
]
ALL_ANNOTATORS = [annotator for spec in COHORT_SPECS for annotator in spec['annotators']]

# Keep the Null subset identical to D_S6_snr_sweep_manual_review.ipynb.
NULL_SELECTION_SEED = 2026072001

# Exact ISO A4 landscape size in inches. Do not use bbox_inches='tight'.
A4_LANDSCAPE = (11.69, 8.27)
PLOTS_PER_PAGE = 50
GRID_ROWS = 7
GRID_COLS = 8
OUTPUT_DPI = 300

print(f'Cohorts: {[spec["cohort_id"] for spec in COHORT_SPECS]}')
print(f'Annotators: {ALL_ANNOTATORS}; plots per annotator: {PLOTS_PER_ANNOTATOR}')
print(f'Pages per annotator: {math.ceil(PLOTS_PER_ANNOTATOR / PLOTS_PER_PAGE)}')

Cohorts: ['cohort1', 'cohort2', 'cohort3']
Annotators: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O']; plots per annotator: 100
Pages per annotator: 2


## Build the stable 480-case master pool

Signal cases are selected by stable S3 `case_id` and `candidate_index`. The 32 Null controls use the same deterministic sampling rule and seed as D_S6. S4 metrics are retained only in the researcher table.

In [3]:
cases = pd.read_csv(CASES_PATH, low_memory=False)
metrics = pd.read_parquet(
    METRICS_PATH,
    columns=['case_id', 'MIC', 'MEV', 'distance_correlation'],
).rename(columns={'distance_correlation': 'dCor'})
if not cases['case_id'].is_unique or not metrics['case_id'].is_unique:
    raise ValueError('case_id must be unique in S3 cases and S4 metrics.')

signal_candidates = cases.loc[~cases['is_null'].astype(bool)].copy()
available_snr = np.sort(signal_candidates['nominal_snr'].dropna().unique())

def nearest_s3_snr(requested: float) -> float:
    return float(available_snr[np.argmin(np.abs(available_snr - float(requested)))])

snr_map = pd.DataFrame({'requested_snr': REQUESTED_SNR})
snr_map['matched_nominal_snr'] = snr_map['requested_snr'].map(nearest_s3_snr)
snr_map['relative_error_percent'] = 100.0 * (
    snr_map['matched_nominal_snr'] - snr_map['requested_snr']
) / snr_map['requested_snr']
if not snr_map['matched_nominal_snr'].is_unique:
    raise ValueError('Two requested SNR values map to the same S3 nominal SNR.')

requested_by_matched = dict(zip(snr_map['matched_nominal_snr'], snr_map['requested_snr']))
family_ids = [family_id for family_id, _ in FAMILY_SPECS]
matched_snr = snr_map['matched_nominal_snr'].to_numpy(dtype=float)

signal_master = signal_candidates[
    signal_candidates['family_id'].isin(family_ids)
    & signal_candidates['variant_level'].eq(SELECTED_VARIANT)
    & signal_candidates['repeat'].isin(SELECTED_REPEATS)
    & signal_candidates['nominal_snr'].isin(matched_snr)
].copy()
signal_master['requested_snr'] = signal_master['nominal_snr'].map(requested_by_matched)
expected_signal = len(FAMILY_SPECS) * len(REQUESTED_SNR) * len(SELECTED_REPEATS)
if len(signal_master) != expected_signal:
    raise ValueError(f'Expected {expected_signal} signal cases; found {len(signal_master)}.')

null_candidates = cases.loc[cases['is_null'].astype(bool)].copy()
if len(null_candidates) < N_NULL:
    raise ValueError(f'Requested {N_NULL} Null cases, but S3 contains only {len(null_candidates)}.')
null_master = null_candidates.sample(n=N_NULL, replace=False, random_state=NULL_SELECTION_SEED).copy()
null_master['requested_snr'] = np.nan

case_master = pd.concat([signal_master, null_master], ignore_index=True, sort=False)
case_master = case_master.merge(metrics, on='case_id', how='left', validate='one_to_one')
if case_master[['MIC', 'MEV', 'dCor']].isna().any().any():
    raise ValueError('At least one selected S3 case is missing S4 metrics.')
if not case_master['case_id'].is_unique:
    raise ValueError('The annotation case pool contains duplicate case_id values.')

def snr_band(value: float) -> str:
    if not np.isfinite(value):
        return 'null'
    if value >= 10.0:
        return 'high_100_to_10'
    if value >= 2.0:
        return 'middle_high_7_to_2'
    if value >= 0.3:
        return 'middle_low_1.5_to_0.3'
    return 'low_0.1'

case_master['snr_band'] = case_master['requested_snr'].map(snr_band)
case_master['sampling_family'] = case_master['family_id'].astype(str)
case_master['sampling_cell'] = np.where(
    case_master['is_null'].astype(bool),
    'null',
    case_master['snr_band'].astype(str) + '|repeat_' + case_master['repeat'].astype(int).astype(str),
)
case_master['pair_key'] = np.where(
    case_master['is_null'].astype(bool),
    case_master['case_id'].astype(str),
    case_master['family_id'].astype(str) + '|snr_' + case_master['requested_snr'].astype(str),
)
case_master = case_master.sort_values(
    ['is_null', 'family_id', 'requested_snr', 'repeat'],
    ascending=[True, True, False, True],
).reset_index(drop=True)
case_master.insert(0, 'annotation_pool_index', np.arange(1, len(case_master) + 1, dtype=int))

expected_total = expected_signal + N_NULL
assert len(case_master) == expected_total == 480
print(f'Case master: {len(case_master)} unique cases = {expected_signal} signal + {N_NULL} Null')
display(snr_map.round({'matched_nominal_snr': 6, 'relative_error_percent': 2}))
display(case_master.groupby(['family_id', 'variant_level'], dropna=False).size().rename('cases').reset_index())

Case master: 480 unique cases = 448 signal + 32 Null


,requested_snr,matched_nominal_snr,relative_error_percent
0,100.0,100.000000,0.00
1,20.0,19.756319,-1.22
2,15.0,15.175880,1.17
3,10.0,10.217053,2.17
4,7.0,6.878558,-1.73
5,5.0,4.946605,-1.07
6,4.0,4.058757,1.47
7,3.0,2.918790,-2.71
8,2.0,1.965055,-1.75
9,1.5,1.509463,0.63


,family_id,variant_level,cases
0,F01,strong,32
1,F03,strong,32
2,F05,strong,32
3,F07,strong,32
4,F13,strong,32
5,F15,strong,32
6,F18,strong,32
7,F19,strong,32
8,F21,strong,32
9,F22,strong,32


## Stratified random sampling and assignment

Each five-person cohort creates one coverage copy of every case, then independently selects 20 distinct, stratified random overlap cases. Allocation balances family and broad SNR bands across annotators. The three cohorts use separate overlap, allocation, and display-order seeds, all of which are stored.

In [4]:
def proportional_quotas(counts: pd.Series, total: int, rng: np.random.Generator) -> pd.Series:
    counts = counts.astype(int)
    if total < 0 or total > int(counts.sum()):
        raise ValueError(f'Cannot sample {total} items from {int(counts.sum())}.')
    if total == 0:
        return pd.Series(0, index=counts.index, dtype=int)
    raw = counts / counts.sum() * total
    quotas = np.floor(raw).astype(int)
    remaining = total - int(quotas.sum())
    fractional = raw - quotas
    tie_break = pd.Series(rng.random(len(counts)) * 1e-9, index=counts.index)
    priority = (fractional + tie_break).sort_values(ascending=False).index.tolist()
    while remaining > 0:
        changed = False
        for key in priority:
            if quotas.loc[key] < counts.loc[key]:
                quotas.loc[key] += 1
                remaining -= 1
                changed = True
                if remaining == 0:
                    break
        if not changed:
            raise RuntimeError('Unable to allocate proportional sampling quotas.')
    return quotas.astype(int)

def stratified_case_sample(pool: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    family_counts = pool.groupby('sampling_family', sort=True).size()
    family_quotas = proportional_quotas(family_counts, n, rng)
    selected_indices = []

    for family_id, family_quota in family_quotas.items():
        family_pool = pool.loc[pool['sampling_family'].eq(family_id)]
        if int(family_quota) == 0:
            continue
        cell_counts = family_pool.groupby('sampling_cell', sort=True).size()
        cell_quotas = proportional_quotas(cell_counts, int(family_quota), rng)
        for cell_name, cell_quota in cell_quotas.items():
            if int(cell_quota) == 0:
                continue
            eligible = family_pool.loc[family_pool['sampling_cell'].eq(cell_name)].index.to_numpy()
            chosen = rng.choice(eligible, size=int(cell_quota), replace=False)
            selected_indices.extend(chosen.tolist())

    if len(selected_indices) != n or len(set(selected_indices)) != n:
        raise RuntimeError('Stratified sample did not produce the requested unique sample size.')
    rng.shuffle(selected_indices)
    return pool.loc[selected_indices].copy()

def allocate_cohort(pool: pd.DataFrame, cohort_spec: dict) -> pd.DataFrame:
    annotators = cohort_spec['annotators']
    allocation_seed = int(cohort_spec['allocation_seed'])
    overlap_seed = int(cohort_spec['overlap_seed'])
    rng = np.random.default_rng(allocation_seed)
    n_overlap = len(annotators) * PLOTS_PER_ANNOTATOR - len(pool)
    overlap_cases = stratified_case_sample(pool, n_overlap, overlap_seed)

    coverage = pool[['case_id']].copy()
    coverage['sampling_role'] = 'coverage'
    overlap = overlap_cases[['case_id']].copy()
    overlap['sampling_role'] = 'random_overlap'
    occurrences = pd.concat([coverage, overlap], ignore_index=True)
    occurrences = occurrences.merge(
        pool[['case_id', 'sampling_family', 'snr_band', 'sampling_cell', 'pair_key', 'repeat']],
        on='case_id', how='left', validate='many_to_one',
    )

    # Family quotas are near-equal (normally 6-7 plots per signal family and 6-7 Null plots
    # per annotator) and sum to exactly 100 for every annotator.
    family_totals = occurrences.groupby('sampling_family', sort=True).size()
    family_quota = pd.DataFrame(0, index=family_totals.index, columns=annotators, dtype=int)
    running_totals = {annotator: 0 for annotator in annotators}
    family_order = family_totals.index.to_numpy(copy=True)
    rng.shuffle(family_order)
    for family_id in family_order:
        total = int(family_totals.loc[family_id])
        base, remainder = divmod(total, len(annotators))
        family_quota.loc[family_id, :] = base
        for annotator in annotators:
            running_totals[annotator] += base
        ranked = sorted(
            annotators,
            key=lambda annotator: (running_totals[annotator], rng.random()),
        )
        for annotator in ranked[:remainder]:
            family_quota.loc[family_id, annotator] += 1
            running_totals[annotator] += 1

    if set(running_totals.values()) != {PLOTS_PER_ANNOTATOR}:
        raise RuntimeError(f'Family quotas did not yield 100 plots each: {running_totals}')

    assignment_rows = []
    for family_id in family_order:
        family_occurrences = occurrences.loc[
            occurrences['sampling_family'].eq(family_id)
        ].reset_index(drop=True)
        target_quota = family_quota.loc[family_id].to_dict()
        multiplicity = family_occurrences.groupby('case_id').size().rename('multiplicity')
        case_table = (
            family_occurrences.drop_duplicates('case_id')
            .set_index('case_id').join(multiplicity).reset_index()
        )
        n_cases_family = len(case_table)
        n_annotators = len(annotators)
        n_variables = n_cases_family * n_annotators

        def variable_index(case_position: int, annotator_position: int) -> int:
            return case_position * n_annotators + annotator_position

        assigned_family = None
        for balance_tolerance in [0, 1, 2]:
            coefficient_rows = []
            lower_bounds = []
            upper_bounds = []

            # Each case is assigned once, or twice when selected for random overlap.
            for case_position, record in case_table.iterrows():
                coefficients = np.zeros(n_variables, dtype=float)
                for annotator_position in range(n_annotators):
                    coefficients[variable_index(case_position, annotator_position)] = 1.0
                required = int(record['multiplicity'])
                coefficient_rows.append(coefficients)
                lower_bounds.append(required)
                upper_bounds.append(required)

            # Exact per-family quotas; their column totals are exactly 100 per annotator.
            for annotator_position, annotator in enumerate(annotators):
                coefficients = np.zeros(n_variables, dtype=float)
                for case_position in range(n_cases_family):
                    coefficients[variable_index(case_position, annotator_position)] = 1.0
                required = int(target_quota[annotator])
                coefficient_rows.append(coefficients)
                lower_bounds.append(required)
                upper_bounds.append(required)

            # Keep broad SNR bands close to proportional within every family.
            for value, value_cases in case_table.groupby('snr_band'):
                case_positions = value_cases.index.to_numpy(dtype=int)
                total_occurrences = int(value_cases['multiplicity'].sum())
                ideal_low = max(0, math.floor(total_occurrences / n_annotators) - balance_tolerance)
                ideal_high = math.ceil(total_occurrences / n_annotators) + balance_tolerance
                for annotator_position in range(n_annotators):
                    coefficients = np.zeros(n_variables, dtype=float)
                    for case_position in case_positions:
                        coefficients[variable_index(case_position, annotator_position)] = 1.0
                    coefficient_rows.append(coefficients)
                    lower_bounds.append(float(ideal_low))
                    upper_bounds.append(float(ideal_high))

            constraint = LinearConstraint(
                np.vstack(coefficient_rows),
                np.asarray(lower_bounds, dtype=float),
                np.asarray(upper_bounds, dtype=float),
            )
            objective_rng = np.random.default_rng(
                allocation_seed + 100 * balance_tolerance + len(assignment_rows)
            )
            result = milp(
                c=objective_rng.random(n_variables) * 1e-3,
                integrality=np.ones(n_variables, dtype=int),
                bounds=Bounds(np.zeros(n_variables), np.ones(n_variables)),
                constraints=constraint,
                options={'time_limit': 10.0},
            )
            if result.success:
                solution = np.rint(result.x).astype(int).reshape(n_cases_family, n_annotators)
                trial_rows = []
                role_rng = np.random.default_rng(allocation_seed + len(assignment_rows))
                for case_position, record in case_table.iterrows():
                    selected_annotators = [
                        annotators[position]
                        for position in np.flatnonzero(solution[case_position])
                    ]
                    role_rng.shuffle(selected_annotators)
                    roles = ['coverage'] + ['random_overlap'] * (len(selected_annotators) - 1)
                    trial_rows.extend({
                        'annotator_id': annotator,
                        'case_id': record['case_id'],
                        'sampling_role': role,
                    } for annotator, role in zip(selected_annotators, roles))
                assigned_family = trial_rows
                break

        if assigned_family is None:
            raise RuntimeError(f'Could not allocate family {family_id} within its balanced quotas.')
        assignment_rows.extend(assigned_family)

    assignments = pd.DataFrame(assignment_rows)
    if assignments.groupby('annotator_id').size().reindex(annotators).to_dict() != {
        annotator: PLOTS_PER_ANNOTATOR for annotator in annotators
    }:
        raise RuntimeError(f"Final {cohort_spec['cohort_id']} annotator totals are not exactly 100.")
    return assignments

cohort_assignment_parts = []
for cohort_position, cohort_spec in enumerate(COHORT_SPECS, start=1):
    cohort_id = cohort_spec['cohort_id']
    annotators = cohort_spec['annotators']
    cohort_assignments = allocate_cohort(case_master, cohort_spec)
    cohort_assignments = cohort_assignments.merge(
        case_master, on='case_id', how='left', validate='many_to_one'
    )
    cohort_assignments['cohort_id'] = cohort_id
    cohort_assignments['overlap_seed'] = int(cohort_spec['overlap_seed'])
    cohort_assignments['allocation_seed'] = int(cohort_spec['allocation_seed'])

    ordered_parts = []
    for local_position, annotator in enumerate(annotators, start=1):
        annotator_rows = cohort_assignments.loc[
            cohort_assignments['annotator_id'].eq(annotator)
        ].copy()
        if len(annotator_rows) != PLOTS_PER_ANNOTATOR:
            raise ValueError(f'Annotator {annotator} has {len(annotator_rows)} assignments, expected 100.')
        display_seed = int(cohort_spec['display_seed']) + local_position
        annotator_rows = annotator_rows.sample(
            frac=1.0, random_state=display_seed
        ).reset_index(drop=True)
        person_number = (cohort_position - 1) * len(annotators) + local_position
        annotator_rows['person_number'] = person_number
        annotator_rows['display_order'] = np.arange(1, PLOTS_PER_ANNOTATOR + 1, dtype=int)
        annotator_rows['plot_code'] = annotator + annotator_rows['display_order'].astype(str)
        annotator_rows['assignment_id'] = [
            f"{cohort_id}_{annotator}_{order:03d}"
            for order in annotator_rows['display_order']
        ]
        annotator_rows['display_order_seed'] = display_seed
        annotator_rows['page_number'] = ((annotator_rows['display_order'] - 1) // PLOTS_PER_PAGE) + 1
        annotator_rows['page_side'] = np.where(
            annotator_rows['page_number'].eq(1), 'front', 'back'
        )
        annotator_rows['page_position'] = ((annotator_rows['display_order'] - 1) % PLOTS_PER_PAGE) + 1
        ordered_parts.append(annotator_rows)

    cohort_map = pd.concat(ordered_parts, ignore_index=True)
    cohort_map['case_assignment_count_in_cohort'] = cohort_map.groupby('case_id')['case_id'].transform('size')
    assert len(cohort_map) == len(annotators) * PLOTS_PER_ANNOTATOR == 500
    assert cohort_map['case_id'].nunique() == len(case_master) == 480
    assert not cohort_map.duplicated(['annotator_id', 'case_id']).any()
    count_distribution = cohort_map.groupby('case_id').size().value_counts().to_dict()
    n_overlap = len(annotators) * PLOTS_PER_ANNOTATOR - len(case_master)
    expected_count_distribution = {1: len(case_master) - n_overlap, 2: n_overlap}
    assert count_distribution == expected_count_distribution, count_distribution
    cohort_assignment_parts.append(cohort_map)

assignment_map = pd.concat(cohort_assignment_parts, ignore_index=True)
assignment_map['case_assignment_count_total'] = assignment_map.groupby('case_id')['case_id'].transform('size')
assignment_map['label'] = np.nan
assignment_map['response_time_seconds'] = np.nan
assignment_map['annotator_comment'] = ''
assignment_map = assignment_map.sort_values(
    ['person_number', 'display_order']
).reset_index(drop=True)

# Put the anonymous sheet code and its original case ID first so the mapping
# opens directly as A1-A100, B1-B100, ..., O1-O100 without horizontal scrolling.
mapping_front_columns = [
    'plot_code', 'case_id', 'annotator_id', 'display_order', 'cohort_id',
    'page_number', 'page_side', 'page_position', 'assignment_id',
]
assignment_map = assignment_map[
    mapping_front_columns
    + [column for column in assignment_map.columns if column not in mapping_front_columns]
]

expected_assignments = len(ALL_ANNOTATORS) * PLOTS_PER_ANNOTATOR
assert len(assignment_map) == expected_assignments == 1500
assert assignment_map['assignment_id'].is_unique
assert assignment_map['case_id'].nunique() == len(case_master) == 480
total_case_counts = assignment_map.groupby('case_id').size()
assert total_case_counts.min() >= len(COHORT_SPECS)
assert total_case_counts.max() <= 2 * len(COHORT_SPECS)

print(f'Assignments: {len(assignment_map)} rows for {len(ALL_ANNOTATORS)} annotators')
display(assignment_map.groupby(['cohort_id', 'annotator_id']).agg(
    plots=('case_id', 'size'), unique_cases=('case_id', 'nunique'),
    null_plots=('is_null', 'sum'), families=('family_id', 'nunique'),
).reset_index())
display(assignment_map.groupby(['cohort_id', 'annotator_id', 'snr_band'], dropna=False).size().unstack(fill_value=0))
display(assignment_map.groupby('case_id').size().value_counts().sort_index().rename('cases_by_total_ratings'))

Assignments: 1500 rows for 15 annotators


,cohort_id,annotator_id,plots,unique_cases,null_plots,families
0,cohort1,A,100,100,6,15
1,cohort1,B,100,100,6,15
2,cohort1,C,100,100,7,15
3,cohort1,D,100,100,7,15
4,cohort1,E,100,100,7,15
5,cohort2,F,100,100,7,15
6,cohort2,G,100,100,7,15
7,cohort2,H,100,100,6,15
8,cohort2,I,100,100,7,15
9,cohort2,J,100,100,7,15


snr_band                high_100_to_10  low_0.1  middle_high_7_to_2  middle_low_1.5_to_0.3  null
cohort_id annotator_id                                                                          
cohort1   A                         25        4                  28                     37     6
          B                         21        5                  28                     40     6
          C                         22        6                  28                     37     7
          D                         22        8                  28                     35     7
          E                         22        5                  28                     38     7
cohort2   F                         21        3                  28                     41     7
          G                         25        5                  28                     35     7
          H                         20        6                  28                     40     6
          I                         22        9                  28                     34     7
          J                         24        5                  28                     36     7
cohort3   K                         21        6                  28                     38     7
          L                         23        6                  28                     36     7
          M                         22        7                  28                     36     7
          N                         25        4                  28                     36     7
          O                         21        5                  28                     40     6

3    428
4     45
5      6
6      1
Name: cases_by_total_ratings, dtype: int64

## Load only the 480 selected scatterplots and render blind A4 sheets

Each annotator receives one exact-size, two-page A4 landscape PDF suitable for duplex printing. The existing A-J combined PDF remains a 20-page file, and a separate K-O combined PDF contains 10 pages. Every page contains 50 black scatterplots in a 7 x 8 grid; the final six panels are blank. Each plot has an approximately 4:3 frame, and only the anonymous plot code is visible.

In [5]:
case_master = case_master.copy()
case_master['loaded_position'] = np.arange(len(case_master), dtype=int)
array_indices = case_master['candidate_index'].to_numpy(dtype=int)
with np.load(POINTS_PATH) as point_store:
    x_master = point_store['x'][array_indices].copy()
    y_master = point_store['y'][array_indices].copy()

position_by_case_id = case_master.set_index('case_id')['loaded_position'].to_dict()
sheet_records = []
combined_pdf_specs = {
    'A_to_J': {
        'cohort_ids': ['cohort1', 'cohort2'],
        'path': PDF_DIR / 'all_annotators_A_to_J_A4_duplex.pdf',
    },
    'K_to_O': {
        'cohort_ids': ['cohort3'],
        'path': PDF_DIR / 'annotators_K_to_O_A4_duplex.pdf',
    },
}
combined_group_by_cohort = {
    cohort_id: group_id
    for group_id, spec in combined_pdf_specs.items()
    for cohort_id in spec['cohort_ids']
}
combined_pdf_writers = {
    group_id: PdfPages(spec['path'])
    for group_id, spec in combined_pdf_specs.items()
}

for cohort_spec in COHORT_SPECS:
    cohort_id = cohort_spec['cohort_id']
    combined_group_id = combined_group_by_cohort[cohort_id]
    combined_pdf_path = combined_pdf_specs[combined_group_id]['path']
    for annotator in cohort_spec['annotators']:
        annotator_rows = assignment_map.loc[
            assignment_map['annotator_id'].eq(annotator)
        ].sort_values('display_order')
        pdf_path = PDF_DIR / f'{cohort_id}_annotator_{annotator}_A4_duplex.pdf'
        preview_paths = []

        with PdfPages(pdf_path) as pdf:
            for page_number in range(1, 1 + math.ceil(PLOTS_PER_ANNOTATOR / PLOTS_PER_PAGE)):
                page = annotator_rows.loc[
                    annotator_rows['page_number'].eq(page_number)
                ].sort_values('page_position')
                fig, axes = plt.subplots(
                    GRID_ROWS, GRID_COLS,
                    figsize=A4_LANDSCAPE,
                    squeeze=False,
                )

                for panel_position, (_, record) in enumerate(page.iterrows()):
                    ax = axes.flat[panel_position]
                    loaded_position = int(position_by_case_id[record['case_id']])
                    ax.scatter(
                        x_master[loaded_position], y_master[loaded_position],
                        s=1.15, alpha=0.58, color='#000000',
                        edgecolors='none', rasterized=True,
                    )
                    ax.set_xlim(-0.02, 1.02)
                    ax.set_xticks([])
                    ax.set_yticks([])
                    ax.set_box_aspect(0.75)
                    ax.set_title(record['plot_code'], fontsize=7.0, fontweight='bold', pad=1.4)
                    for spine in ax.spines.values():
                        spine.set_color('#777777')
                        spine.set_linewidth(0.45)

                for unused_ax in axes.flat[len(page):]:
                    unused_ax.set_visible(False)

                fig.subplots_adjust(
                    left=0.015, right=0.992, bottom=0.025, top=0.980,
                    wspace=0.06, hspace=0.26,
                )
                pdf.savefig(fig, dpi=OUTPUT_DPI, facecolor='white')
                combined_pdf_writers[combined_group_id].savefig(
                    fig, dpi=OUTPUT_DPI, facecolor='white'
                )
                png_path = PREVIEW_DIR / f'{cohort_id}_annotator_{annotator}_page_{page_number}.png'
                fig.savefig(png_path, format='png', dpi=OUTPUT_DPI, facecolor='white')
                preview_paths.append(str(png_path))
                plt.close(fig)

        sheet_records.append({
            'cohort_id': cohort_id,
            'annotator_id': annotator,
            'n_plots': len(annotator_rows),
            'n_pages': len(preview_paths),
            'plots_per_page': PLOTS_PER_PAGE,
            'pdf_path': str(pdf_path),
            'preview_page_1_path': preview_paths[0],
            'preview_page_2_path': preview_paths[1],
            'page_width_inches': A4_LANDSCAPE[0],
            'page_height_inches': A4_LANDSCAPE[1],
            'dpi': OUTPUT_DPI,
            'combined_pdf_path': str(combined_pdf_path),
        })
        print(f'Saved: {pdf_path}')

for group_id, writer in combined_pdf_writers.items():
    writer.close()
    print(f"Saved combined {group_id.replace('_', '-')} PDF: {combined_pdf_specs[group_id]['path']}")

sheet_manifest = pd.DataFrame(sheet_records)
display(sheet_manifest)

Saved: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/pdf/cohort1_annotator_A_A4_duplex.pdf


Saved: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/pdf/cohort1_annotator_B_A4_duplex.pdf


Saved: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/pdf/cohort1_annotator_C_A4_duplex.pdf


Saved: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/pdf/cohort1_annotator_D_A4_duplex.pdf


Saved: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/pdf/cohort1_annotator_E_A4_duplex.pdf


...

Saved: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/pdf/cohort3_annotator_N_A4_duplex.pdf


Saved: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/pdf/cohort3_annotator_O_A4_duplex.pdf


Saved combined A-to-J PDF: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/pdf/all_annotators_A_to_J_A4_duplex.pdf


Saved combined K-to-O PDF: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/pdf/annotators_K_to_O_A4_duplex.pdf


,cohort_id,annotator_id,n_plots,n_pages,plots_per_page,pdf_path,preview_page_1_path,preview_page_2_path,page_width_inches,page_height_inches,dpi,combined_pdf_path
0,cohort1,A,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...
1,cohort1,B,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...
2,cohort1,C,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...
3,cohort1,D,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...
4,cohort1,E,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...
5,cohort2,F,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...
6,cohort2,G,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...
7,cohort2,H,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...
8,cohort2,I,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...
9,cohort2,J,100,2,50,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,/Users/mimi/Documents/Code/Github/GSA-work/exp...,11.69,8.27,300,/Users/mimi/Documents/Code/Github/GSA-work/exp...


## Export researcher mapping and annotator response templates

Only the PDFs and the two-column blank label templates should be shared. The case master and assignment map contain unblinded metadata and remain researcher-only.

In [6]:
case_master_export = case_master.drop(columns=['loaded_position']).copy()
case_master_export['source_data_notebook'] = 'D_S3_snr_sweep_generation.ipynb'
case_master_export['source_metrics_notebook'] = 'D_S4_snr_sweep_metrics.ipynb'

stale_case_master_path = RESEARCHER_DIR / 'annotation_case_master_406.csv'
if stale_case_master_path.exists():
    stale_case_master_path.unlink()
case_master_path = RESEARCHER_DIR / 'annotation_case_master_480.csv'
cohort_map_paths = {
    spec['cohort_id']: RESEARCHER_DIR / f"{spec['cohort_id']}_assignment_map.csv"
    for spec in COHORT_SPECS
}
assignment_map_path = RESEARCHER_DIR / 'all_annotators_assignment_map.csv'
sheet_manifest_path = RESEARCHER_DIR / 'annotation_sheet_manifest.csv'
metadata_path = RESEARCHER_DIR / 'generation_metadata.json'

case_master_export.to_csv(case_master_path, index=False)
for cohort_id, cohort_path in cohort_map_paths.items():
    assignment_map.loc[assignment_map['cohort_id'].eq(cohort_id)].to_csv(cohort_path, index=False)
assignment_map.to_csv(assignment_map_path, index=False)
sheet_manifest.to_csv(sheet_manifest_path, index=False)

for annotator in ALL_ANNOTATORS:
    template = assignment_map.loc[
        assignment_map['annotator_id'].eq(annotator),
        ['plot_code', 'display_order'],
    ].sort_values('display_order').drop(columns='display_order').copy()
    template['label'] = ''
    template.to_csv(TEMPLATE_DIR / f'annotator_{annotator}_labels.csv', index=False)

metadata = {
    'design': 'three independent complete-coverage cohorts with independent stratified overlap samples',
    'cohorts': COHORT_SPECS,
    'annotators': ALL_ANNOTATORS,
    'plots_per_annotator': PLOTS_PER_ANNOTATOR,
    'pages_per_annotator': int(math.ceil(PLOTS_PER_ANNOTATOR / PLOTS_PER_PAGE)),
    'plots_per_page': PLOTS_PER_PAGE,
    'combined_pdfs': {
        group_id: str(spec['path'])
        for group_id, spec in combined_pdf_specs.items()
    },
    'unique_case_pool_size': int(len(case_master)),
    'signal_cases': int((~case_master['is_null'].astype(bool)).sum()),
    'null_cases': int(case_master['is_null'].astype(bool).sum()),
    'requested_snr': REQUESTED_SNR,
    'selected_variant': SELECTED_VARIANT,
    'selected_repeats': SELECTED_REPEATS,
    'seeds': {
        'null_selection': NULL_SELECTION_SEED,
        **{
            spec['cohort_id']: {
                key: value for key, value in spec.items() if key.endswith('_seed')
            }
            for spec in COHORT_SPECS
        },
    },
    'annotator_visible_fields': ['plot_code', 'scatterplot'],
    'researcher_mappings': {
        'all_annotators': str(assignment_map_path),
        **{cohort_id: str(path) for cohort_id, path in cohort_map_paths.items()},
    },
}
with metadata_path.open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2, sort_keys=True)

print('Researcher-only outputs')
print(f'  Case master:    {case_master_path}')
for cohort_id, cohort_path in cohort_map_paths.items():
    print(f'  {cohort_id} map: {cohort_path}')
print(f'  Combined map:   {assignment_map_path}')
print(f'  Sheet manifest: {sheet_manifest_path}')
print(f'  Metadata:       {metadata_path}')
print(f'Annotator PDFs:   {PDF_DIR}')
for group_id, spec in combined_pdf_specs.items():
    print(f"Combined {group_id.replace('_', '-')} PDF: {spec['path']}")
print(f'Label templates:  {TEMPLATE_DIR}')

Researcher-only outputs
  Case master:    /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/researcher_only/annotation_case_master_480.csv
  cohort1 map: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/researcher_only/cohort1_assignment_map.csv
  cohort2 map: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/researcher_only/cohort2_assignment_map.csv
  cohort3 map: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/researcher_only/cohort3_assignment_map.csv
  Combined map:   /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/researcher_only/all_annotators_assignment_map.csv
  Sheet manifest: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S7_annotation_sheets/researcher_only/annotation_sheet_manifest.csv
  Metadata:       /User